# Ottimizzazione delle prestazioni di una rete neurale per il settore food

## Introduzione
**GourmetAI Inc.**, una rinomata azienda nel settore della tecnologia alimentare, si trova ad affrontare sfide crescenti nel migliorare l'accuratezza e l'efficienza dei sistemi di classificazione delle immagini di cibo. La necessità di fornire ai clienti soluzioni avanzate e di alta qualità per identificare e categorizzare correttamente i cibi è essenziale per migliorare l'esperienza utente e ottimizzare i processi aziendali.

---

## Benefici del Progetto
L'implementazione di tecniche avanzate di deep learning per la classificazione delle immagini di cibo offre numerosi vantaggi:

- **Miglioramento dell'Esperienza Utente**: Un sistema preciso e affidabile migliora significativamente l'esperienza degli utenti nelle applicazioni di riconoscimento di immagini di cibo, fornendo risultati rapidi e accurati.
- **Ottimizzazione dei Processi Aziendali**: Automatizzare la classificazione delle immagini riduce il tempo e le risorse necessarie per identificare e categorizzare i cibi, migliorando l'efficienza operativa complessiva.
- **Innovazione Tecnologica**: Utilizzare tecniche avanzate di deep learning promuove l'innovazione all'interno dell'azienda, consentendo di affrontare sfide complesse nel settore del riconoscimento delle immagini.
- **Obiettivi di Business**: Migliorare le performance dei modelli di classificazione aiuta GourmetAI Inc. a soddisfare gli obiettivi di business, consolidando la propria posizione come leader nel settore della tecnologia alimentare.

---

## Dettagli del Progetto
GourmetAI Inc. ha richiesto lo sviluppo di un modello avanzato di classificazione di immagini di cibo utilizzando tecniche di deep learning. Il progetto si baserà sul dataset **Food Classification**, arricchito con tecniche di augmentation per migliorare la diversità e la qualità dei dati disponibili.

### Obiettivi del Progetto
1. **Strategie di Augmentation**: Implementare diverse tecniche di augmentation per arricchire il dataset, migliorando la variabilità e la qualità dei dati.
2. **Divisione del Dataset**: Suddividere il dataset in trainset, valset e testset per garantire un'adeguata formazione e validazione del modello.
3. **Architetture di Rete e Transfer Learning**: Selezionare e implementare una o più architetture di rete neurale adatte al problema, utilizzando il transfer learning per sfruttare modelli pre-addestrati.
4. **Fine Tuning e Scelta degli Hyperparameters**: Creare un classificatore personalizzato, scegliere gli hyperparameters e ottimizzare il modello attraverso processi di training e validation.
5. **Validation e Regolarizzazione**: Utilizzare tecniche di validation per migliorare la scelta degli hyperparameters e risolvere potenziali problemi con tecniche di regolarizzazione.
6. **Test Finale**: Eseguire un test finale per verificare le capacità di generalizzazione del modello e raggiungere le performance desiderate.

---

## Fasi del Progetto
1. **Sviluppo delle Strategie di Augmentation**: Esplorazione e implementazione di tecniche di augmentation per arricchire il dataset.
2. **Preparazione del Dataset**: Divisione del dataset in trainset, valset e testset; preparazione degli strumenti per l'utilizzo delle immagini come input.
3. **Selezione e Implementazione delle Architetture**: Scelta e implementazione di architetture di rete neurale, applicando il transfer learning.
4. **Fine Tuning e Scelta degli Hyperparameters**: Creazione di un classificatore personalizzato, selezione degli hyperparameters e ottimizzazione del modello.
5. **Validation e Regolarizzazione**: Ripetizione del training con tecniche di validation e regolarizzazione per migliorare le performance del modello.
6. **Test Finale**: Esecuzione di un test finale per valutare le capacità di generalizzazione del modello e confrontare i risultati con le aspettative.

---

## Motivazione del Progetto
Per GourmetAI Inc., la precisione nella classificazione delle immagini di cibo rappresenta un requisito fondamentale. Migliorare l'efficacia dei sistemi di classificazione non solo migliora l'esperienza utente e ottimizza i processi aziendali, ma consente anche di consolidare la leadership nel settore della tecnologia alimentare attraverso l'innovazione tecnologica.

Con questo progetto, GourmetAI Inc. mira a sviluppare e implementare un sistema avanzato di classificazione di immagini di cibo, utilizzando tecniche all'avanguardia di deep learning per raggiungere performance superiori e soddisfare le esigenze specifiche del mercato.

---

## Dataset
Il dataset è disponibile al seguente link: [Food Classification Dataset](https://proai-datasets.s3.eu-west-3.amazonaws.com/dataset_food_classification.zip)

# Implementazione

In [14]:
%pip install --quiet torch torchvision albumentations numpy matplotlib seaborn pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [15]:
#Dependencies import

# Standard library 
import os
import random
import requests
import zipfile

from dataclasses import dataclass, field

#PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data_utils
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchvision.transforms as transforms
import torchvision.datasets as datasets

#Metrics
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score, MulticlassPrecision, MulticlassRecall

#Data management and compute
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

#Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [ ]:
#DNNHelper module download

url = "https://raw.githubusercontent.com/crypto-infinity/proai/refs/heads/course6-pytorch/course6-pytorch/dnnhelper/__init__.py"
directory = "./dnnhelper"
file_path = os.path.join(directory, "__init__.py")

os.makedirs("./dnnhelper", exist_ok = True)
response = requests.get(url)

if response.status_code == 200:
    with open(file_path, "wb") as file:
        file.write(response.content)
    from dnnhelper import Experiment, EarlyStopping
else:
    raise Exception(f"Errore durante il download del file: {response.status_code}")

In [ ]:
#Dataset download and extraction

url = "https://proai-datasets.s3.eu-west-3.amazonaws.com/dataset_food_classification.zip"
directory = "./dataset"
file_path = os.path.join(directory, "food_classification.zip")
directory_to_extract_to = "./dataset/food_classification"

os.makedirs("./dataset", exist_ok = True)
response = requests.get(url)

if response.status_code == 200:
    with open(file_path, "wb") as file:
        file.write(response.content)
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(directory_to_extract_to)

else:
    raise Exception(f"Errore durante il download del file: {response.status_code}")

In [11]:
#PyTorch device setup

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Random seed for reproducibility

seed = 56

random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Import del dataset

In [ ]:
#Dataset import

